# Student Depression Prediction
**Matteo Pietro Di Pilato · BSc in Artificial Intelligence · Machine Learning, Module 1 (2024/25)**  
Technical review · November 2026

This notebook revisits the final L1-regularized logistic regression from the February 2025 submission. The original notebook explored Perceptron, logistic regression, random forest and XGBoost, with alternative resampling and dimensionality-reduction configurations. Its 5-fold outer and 2-fold inner search produced a mean outer F1 of 0.868. The final logistic regression reported F1 0.879, precision 0.865, recall 0.894 and accuracy 0.856 on the 20% split.

The review corrects the sleep-duration encoding and computes each confusion matrix from the estimator named in its title. It adds a majority-class baseline and a comparison without the suicidal-thought question. The split and deterministic cleaning rules are inherited from the original study. The split was already inspected in 2025, so this is a retrospective comparison on the same dataset.

## 1. Data and setup

The repository includes version 1 of the [Student Depression Dataset](https://www.kaggle.com/datasets/hopesb/student-depression-dataset) as `StudentDepressionDataset.csv`. Run all cells from the repository root. Set `DATA_PATH` to the local filename if you saved it elsewhere. The tested environment used Python 3.14.4, pandas 3.0.2, NumPy 2.4.3, SciPy 1.17.1 and scikit-learn 1.8.0.

The outcome is the CSV's binary `Depression` label. The data card does not specify how this label was established. This exercise classifies the dataset label and does not evaluate diagnosis or future onset.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 30
DATA_PATH = Path("StudentDepressionDataset.csv")
REFERENCE_SHA256 = "a3e159e246789be5a61dcc7b4290fbc238f0140f3d95727585af50a61fd35404"
raw_data = pd.read_csv(DATA_PATH)
csv_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print(f"Source: {len(raw_data):,} rows and {raw_data.shape[1]} columns")
print(f"CSV SHA-256: {csv_hash}")
if csv_hash != REFERENCE_SHA256:
    print("CSV differs from the reference download. Check the dataset version before comparing scores.")

Source: 27,901 rows and 18 columns
CSV SHA-256: a3e159e246789be5a61dcc7b4290fbc238f0140f3d95727585af50a61fd35404


## 2. Historical cleaning and split

The original protocol masked the **same** random 10% of rows in four fields (gender, city, dietary habits and degree) as a missing-data exercise. It grouped cities and degrees, treated ambiguous categories as missing and removed rows with missing financial stress or sleep duration. This review preserves those rules. Jaipur is assigned once to the group it effectively received in the original code.

The stratified 80/20 split uses seed 30. Learned transformations, including mode imputation, one-hot encoding and scaling, are fitted inside each training fold during tuning.

In [2]:
data = raw_data.rename(columns={
    'Academic Pressure': 'Academic_pres',
    'Study Satisfaction': 'Study_sat',
    'Sleep Duration': 'Sleep_dur',
    'Dietary Habits': 'Diet_habits',
    'Have you ever had suicidal thoughts ?': 'Suic_thought',
    'Work/Study Hours': 'Work_stud_hr',
    'Financial Stress': 'Fin_stress',
    'Family History of Mental Illness': 'Family_hist'
})

data = data.drop(columns=["id", "Profession", "Work Pressure", "Job Satisfaction"])

mask_columns = ["Gender", "City", "Diet_habits", "Degree"]
mask_rows = data.sample(frac=0.10, random_state=SEED).index
data.loc[mask_rows, mask_columns] = np.nan

data['Sleep_dur'] = data['Sleep_dur'].replace('Others', np.nan)
data['Diet_habits'] = data['Diet_habits'].replace('Others', np.nan)
data['Degree'] = data['Degree'].replace('Others', np.nan)

degree_mapping = {
    'Pre-University': ['Class 12'],
    'Undergraduate Degree': ['B.Com', 'BBA', 'BSc', 'BCA', 'B.Tech', 'BE', 'B.Pharm', 'BA', 'LLB', 'B.Arch', 'BHM', 'MBBS', 'B.Ed'],
    'Postgraduate Degree': ['M.Com', 'MBA', 'MSc', 'MCA', 'M.Tech', 'ME', 'M.Pharm', 'MA', 'LLM', 'MHM', 'MD', 'M.Ed'],
    'PhD': ['PhD']
}

degree_to_category = {degree: category for category, degrees in degree_mapping.items() for degree in degrees}

data['Degree'] = data['Degree'].map(degree_to_category)

city_mapping = {
    'Major Metropolitan Cities': [
        'Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 
        'Kolkata', 'Chennai', 'Ahmedabad', 'Pune'
    ],

    'Industrial and Commercial Cities': [
        'Surat', 'Lucknow', 'Patna', 'Visakhapatnam', 
        'Bhopal', 'Ludhiana', 'Nagpur', 'Indore', 'Kanpur', 
        'Vadodara', 'Ghaziabad', 'Faridabad'
    ],

    'Suburban & Satellite Cities': [
        'Thane', 'Vasai-Virar', 'Kalyan', 'Meerut', 'Rajkot', 
        'Nashik', 'Varanasi'
    ],

    'Other Cities': [
        'Agra', 'Srinagar', 'Jaipur'
    ],

    'Miscellaneous / Outliers':[
        'Saanvi', 'Bhavna', 'Harsha', 'Less Delhi', 'M.Tech', 
        '3.0', 'Less than 5 Kalyan', 'Mira', 'Gaurav', 'Harsh', 
        'Reyansh', 'Kibara', 'Rashi', 'ME', 'M.Com', 'Nalyan', 
        'City', 'Mihir', 'Nalini', 'Nandini', 'Khaziabad'
    ]
}

city_to_category = {city: category for category, cities in city_mapping.items() for city in cities}

data['City'] = data['City'].map(city_to_category)
data['City'] = data['City'].replace('Miscellaneous / Outliers', np.nan)

data = data.dropna(subset=["Fin_stress", "Sleep_dur"])

In [3]:
X = data.drop(columns="Depression")
y = data["Depression"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
split_counts = pd.DataFrame({
    "All retained": y.value_counts().sort_index(),
    "Training": y_train.value_counts().sort_index(),
    "Hold-out": y_test.value_counts().sort_index(),
})
split_counts.index.name = "Depression label"
display(split_counts)

,All retained,Training,Hold-out
Depression label,,,
0,11554,9243,2311
1,16326,13061,3265


## 3. Fit the baseline and the two logistic models

Sleep duration now increases in the order under 5, 5–6, 7–8 and over 8 hours. Both logistic models use L1 regularization and select `C` independently on the training data by 5-fold stratified cross-validation with positive-class F1. The majority baseline predicts the most frequent training label. Both logistic models use the default decision threshold.

In [4]:
def logistic_pipeline(binary_features):
    nominal = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ])
    sleep = Pipeline([
        ("encoder", OrdinalEncoder(categories=[[
            "Less than 5 hours", "5-6 hours", "7-8 hours", "More than 8 hours"
        ]])),
        ("scaler", StandardScaler()),
    ])
    diet = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[["Unhealthy", "Moderate", "Healthy"]])),
        ("scaler", StandardScaler()),
    ])
    preprocess = ColumnTransformer([
        ("numeric", StandardScaler(), [
            "Age", "Academic_pres", "CGPA", "Study_sat", "Work_stud_hr", "Fin_stress"
        ]),
        ("nominal", nominal, ["Gender", "City", "Degree"]),
        ("binary", OneHotEncoder(drop="first", handle_unknown="ignore"), binary_features),
        ("sleep", sleep, ["Sleep_dur"]),
        ("diet", diet, ["Diet_habits"]),
    ], sparse_threshold=0)
    return Pipeline([
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(
            solver="saga", l1_ratio=1.0, max_iter=5000, random_state=SEED
        )),
    ])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
c_grid = {"classifier__C": np.logspace(-3, 2, 11)}
fits = {"Majority baseline": DummyClassifier(strategy="most_frequent").fit(X_train, y_train)}
feature_sets = {
    "L1 logistic (all predictors)": ["Family_hist", "Suic_thought"],
    "L1 logistic (without suicidal-thought item)": ["Family_hist"],
}
for name, binary_features in feature_sets.items():
    search = GridSearchCV(
        logistic_pipeline(binary_features), c_grid, scoring="f1", cv=cv, error_score="raise"
    )
    search.fit(X_train, y_train)
    fits[name] = search.best_estimator_
    print(f"{name}: selected C = {search.best_params_['classifier__C']:.4g}")

L1 logistic (all predictors): selected C = 31.62
L1 logistic (without suicidal-thought item): selected C = 0.1


## 4. Compare on the shared hold-out

The hold-out was excluded from fitting and tuning in this review. Its results had already been inspected during the 2025 project. Confusion matrices are for the hold-out only, with true labels in rows and predictions in columns.

In [5]:
metrics = {}
matrices = {}
for name, estimator in fits.items():
    prediction = estimator.predict(X_test)
    probability = estimator.predict_proba(X_test)[:, 1]
    metrics[name] = {
        "F1 (label 1)": f1_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction),
        "Accuracy": accuracy_score(y_test, prediction),
        "Balanced accuracy": balanced_accuracy_score(y_test, prediction),
        "ROC AUC": roc_auc_score(y_test, probability),
    }
    matrices[name] = confusion_matrix(y_test, prediction, labels=[0, 1])

results = pd.DataFrame.from_dict(metrics, orient="index")
display(results.round(3))
display(pd.DataFrame.from_dict(
    {name: matrix.ravel() for name, matrix in matrices.items()},
    orient="index",
    columns=["True 0 → Pred 0", "True 0 → Pred 1", "True 1 → Pred 0", "True 1 → Pred 1"],
))

,F1 (label 1),Precision,Recall,Accuracy,Balanced accuracy,ROC AUC
Majority baseline,0.739,0.586,1.000,0.586,0.500,0.500
L1 logistic (all predictors),0.882,0.866,0.898,0.859,0.851,0.929
L1 logistic (without suicidal-thought item),0.838,0.819,0.858,0.806,0.795,0.880


,True 0 → Pred 0,True 0 → Pred 1,True 1 → Pred 0,True 1 → Pred 1
Majority baseline,0,2311,0,3265
L1 logistic (all predictors),1859,452,332,2933
L1 logistic (without suicidal-thought item),1694,617,464,2801


## 5. Interpretation

The paired comparison measures the change when the suicidal-thought item is unavailable under this protocol. The question concerns a related mental-health outcome and its timing relative to the target is undocumented. The estimates are specific to this dataset and split. A prospective application would require an explicit prediction time, predictors available at that time, a documented target and independent validation.

The original exploratory search is preserved in the submitted notebook. It included invalid oversampling candidates and an XGBoost learning-rate distribution whose third positional argument shifted the intended range. Those choices were part of the exploratory search. The archived final estimator and the two logistic fits here use neither oversampling nor dimensionality reduction.